### GD_Landsat_01_Setup defines regions of interest for SEAN glaciers.
Region of interest is defined by a box drawn on the map LonMin, LatMin, LonMax, LatMax.  
Then the region is converted to UTM8N coordinates, turned into a rectangle aligned with the baseline there.  
Then converted back into WGS84 and stored as x1,y1 through x5,y5.  
Saved as glacierPropsLandsat.csv and in glacierPropsLandsat.shp  
Don't modify x,y manually - only adjust the Lon,Lat set.  
PROBLEM: not all Landsat use UTM8N, some use 7. Note that each image band's metadata has crs info e.g. 'crs': 'EPSG:32608', 'crs_transform': [30, 0, 289785, 0, -30, 6631515]}  
See also: GD_Landsat_02_Download.ipynb  

NOTE: 2 spaces at end of markdown line gives new line.  
TODO: create additional rows for whole glaciers (based on RGI outlines instead of manual boxes)  
TODO: add whole-glacier rows (perhaps combining adjacent glaciers) - complicated by large glaciers that cover multiple Landsat scenes  
TODO: combine Alsek GP into one.  

In [ ]:
import geemap
import pandas as pd
import ee
import os
import pyproj
from shapely.geometry import box
from shapely.ops import transform
import geopandas as gpd

In [ ]:
# Initialize Earth Engine
ee.Authenticate()
ee.Initialize()

In [ ]:
#if it doesn't exist, create glacierPropsLandsat.csv
folder_base = r'C:\Users\andyb\Documents\U\SEAN_Glacier-Dynamics' #os.path.join()
#folder_shp = r'C:\Users\andyb\Documents\U\GEE-Courses\data'  #get away from this...
folder_land = r'C:\Users\andyb\Documents\U\GlacierLandsat'

file_path=os.path.join(folder_base,'glacierPropsLandsat.csv')
if os.path.exists(file_path):
    print(f"File {file_path} already exists.")
else: # If file doesn't exist, save the DataFrame to a new file
    glaciers=pd.read_csv(os.path.join(folder_base,'glacierProps.csv'))
    #print(glaciers)
    glacierLandsat=glaciers.iloc[:,0:3]
    glacierLandsat['Region']='Terminus'
    glacierLandsat['LonCenter']=glaciers['leafletLon']
    glacierLandsat['LatCenter']=glaciers['leafletLat']
    glacierLandsat['LonMin']=glaciers['leafletLon']-.1
    glacierLandsat['LonMax']=glaciers['leafletLon']+.1
    glacierLandsat['LatMin']=glaciers['leafletLat']-.1
    glacierLandsat['LatMax']=glaciers['leafletLat']+.1
    #print(glacierLandsat)
    glacierLandsat.to_csv(os.path.join(folder_base,'glacierPropsLandsat.csv'), index=False)
    print(f"File {file_path} created.")

#To add new glaciers or revise, temporarily rename glacierPropsLandsat.csv, add to glacierProps.csv at least these:
#   Name,GLIMS,RGI60.01,leafletLon,leafletLat,leafletZoom
#   Then run this and copy/paste together a new glacierPropsLandsat.csv
#   Then iterate - rerun this file through shp export, then rerun with new shp files to check...

#recalculate Center from Max/Min? No, better to leave consistent with leafletLat and Lon.

In [ ]:
# Load glacierPropsLandsat.csv
df = pd.read_csv(file_path)

dfnames=list(df.columns) #or can get an index object with just df.columns

#dflabel was an attempt to label the map. It failed, so commenting here.
#dflabel=df[['Name','LonCenter','LatCenter']].copy()
#dflabel.rename(columns={'LonCenter': 'Longitude', 'LatCenter': 'Latitude'}, inplace=True)
#FIXED: This now gives: SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame
#FIX = add .copy() to explicitly make dflabel a new independent data frame.

In [ ]:
#df[['Name','Region']]
df.iloc[8]

In [ ]:
#add whole-glacier rows to glacierPropsLandsat.csv
#load RGI60.01, set center coords, get bounding box for whole glacier outline (plus a little buffer? e.g. for Johns Hopkins growth?)
#... 0.001 degrees lat is ~100 m. 
folder_rgi = r'C:\Users\andyb\Documents\U\GlacierOutlines\nsidc0770_01.rgi60.Alaska'
rgi = gpd.read_file(os.path.join(folder_rgi, "01_rgi60_Alaska.shp")) #r'C:\Users\andyb\Documents\U\GEE-Courses\data\glacierPropsLandsat.shp')
print("Available columns:", rgi.columns.tolist())

#print(rgi[rgi['Name']=='Gilkey Glacier'])
#print(rgi[rgi['Name'].isin(['Mendenhall Glacier','Gilkey Glacier','Lemon Creek Glacier'])]) 
#Added these GLIMSId's to glacierPropsLandsat - note that Margerie Glims Easting didn't match, so maybe I'm using a different version here? Johns Hopkins matched

In [ ]:
#create new row of RGI for AlsekGP. Alsek 23654 GP 20985.
# 2. Get the geometries of shapes 2 and 3 (which are at indices 2 and 3)
geometry_2 = rgi[rgi['RGIId']=='RGI60-01.23654'].geometry
geometry_3 = rgi[rgi['RGIId']=='RGI60-01.20985'].geometry




#working here



############# got stuck with joining these ##############################

In [ ]:
print(geometry_2)

In [ ]:
# 3. Create a new shape with the union of shapes 2 and 3
# The .union() method from shapely (used by geopandas) performs the union operation
# You can use the .unary_union method if you have multiple geometries to union at once
new_shape_geometry = geometry_2.union(geometry_3)
print(new_shape_geometry)

In [ ]:
rgi.crs

In [ ]:
# 4. Create a new GeoDataFrame for the new shape to prepare for concatenation
# You can copy the attribute data from one of the original rows if needed,
# but here a new row with default/empty attributes is created except for the geometry
# Ensure columns match the original GeoDataFrame
new_row_data = {}
for col in rgi.columns:
    if col == rgi.geometry.name: # Use the actual geometry column name
        new_row_data[col] = new_shape_geometry
    else:
        new_row_data[col] = None # Assign None to other attributes, or specific values

# Create a single-row GeoDataFrame
new_rgi_row = gpd.GeoDataFrame([new_row_data], crs=rgi.crs)

# 5. Add the new shape (row) to the end of the object
# The pandas.concat function is the recommended way to append GeoDataFrames
rgi = pd.concat([rgi, new_rgi_row], ignore_index=True)

In [ ]:
#get lat/long bounds of shapes
rgib=rgi.geometry.bounds
rgib.head()

In [ ]:
# Join df2 to df1
rgi=rgi.join(rgib)
rgi.head()

#get bounding boxes (not needed)
#rgibb = rgi.geometry.envelope

# 3. (Optional) Create a new GeoDataFrame with these bounding boxes
# This keeps the original data (attributes) and replaces geometry with boxes
#bbox_gdf = rgi.copy()
#bbox_gdf['geometry'] = rgibb

In [ ]:
#df2 is a copy, preparing for merge
df2=df.copy()
df2['Region']='Whole'
df2['RGIId']='RGI60-01.' + df2['RGI60.01'].astype(str).str.zfill(5)
df2.tail()

In [ ]:
df3=pd.merge(df2,rgi,on='RGIId',how='left')

In [ ]:
df3[['Name_x','Name_y']]
print(df3)

In [ ]:
df3.columns.to_list()

In [ ]:
asdf = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6], "C": [7, 8, 9]})
asdf.rename(columns={'A': 'Alpha', 'B': 'Beta'}, inplace=True)
print(asdf)

asdf.rename(columns={'Beta': 'Alpha'},inplace=True) #allows duplicate names
print(asdf)

asdf['Alpha']

In [ ]:
asdf.rename(columns={'A': 'Alpha', 'B': 'Beta'}, inplace=True)
asdf.drop(columns=['Column_A', 'Column_B'], inplace=True)

In [ ]:
LonCenter    -138.025
LatCenter       59.13
LonMin       -138.219
LonMax       -137.866
LatMin         58.954
LatMax         59.248

In [ ]:
#matched_rows = df[df['model'].str.contains('Mac')]
#print(matched_rows)

In [ ]:
# Initialize a geemap Map
Map = geemap.Map()
Map.add_basemap('SATELLITE')

# Initialize an empty list to store features
features = []
glacierPts = []

# Loop through each region in the CSV and add it to the map
for index, row in df.iterrows():
    # Create a bounding box geometry
    region = ee.Geometry.Rectangle([row['LonMin'], row['LatMin'], row['LonMax'], row['LatMax']])
    # Create a feature with the region name
    feature = ee.Feature(region, {'name': row['Name']+" "+row['Region']})
    # Append to features list
    features.append(feature)
    # Add the region to the map with a unique color or style
    #Map.addLayer(feature, {'color': 'blue'}, row['Name']+" "+row['Region']) #better to add collection below

    glacierPt = ee.Geometry.Point(row['LonCenter'],row['LatCenter'])
    glacierPts.append(glacierPt)

# Add a marker at the points (optional, for visualization)
gpts = ee.FeatureCollection(glacierPts)
Map.addLayer(gpts, {'color': 'red'}, 'Points')

# Convert the list to a FeatureCollection
fc = ee.FeatureCollection(features)
#NOTE: plotting works if we overwrite fc with fc.style() but saving no longer works

# Apply the styling to the FeatureCollection
fcs = fc.style(**{ #asterisk passes in as arguments instead of dictionary
    'color': 'FF0000',      # Red outline
    'width': 2,             # 2 pixels wide
    'lineType': 'dotted',
    'fillColor': 'FF000033' # 00000000 Fully transparent fill
    }) #(**style_params)
#Map.addLayer(fcs, {}, 'All roi') #turned off in favor of GeoDataFrame version below since it shows names on hover

#NOTE: giving up on these labels for now
#FAILS: Map.add_labels(dflabel,column='Name',font_size="10pt",font_color="red",font_family="arial",font_weight="bold")
# Add labels using the 'label' property
#FAILS: Map.addLayer(fc,{'textColor': 'red', 'fontSize': 12},'Polygon Labels',shown=True,labelProperty='Name')  # Property to use for labels

# Center the map on the Nth region
n=0
Map.centerObject(ee.Geometry.Rectangle([df['LonMin'][n], df['LatMin'][n], df['LonMax'][n], df['LatMax'][n]]), zoom=7)

In [ ]:
# Load GeoDataFrame version
gdf = gpd.read_file(os.path.join(folder_land, "glacierPropsLandsat.shp")) #r'C:\Users\andyb\Documents\U\GEE-Courses\data\glacierPropsLandsat.shp')
print("Available columns:", gdf.columns.tolist())
Map.add_gdf(gdf, layer_name='Glaciers from shp') #this is automatically adding glacier name to box

#try:
#    Map.add_labels(gdf, column='name',font_size='12pt', font_color='blue', font_family='arial', font_weight='bold')
#except ValueError as e:
#    print("Error:", e)
#    print("Please choose a column from:", gdf.columns.tolist())
#AKB Note: not sure why this is failing. I've tried a variety of things...

# Map.to_html('map_with_labels.html')  # To save as HTML
# Map.save('map.png')

In [ ]:
Map

In [ ]:
# Draw any shapes on the map using the Drawing tools before executing this code block
if Map.user_roi is not None:
    roi = Map.user_roi
    rinfo=roi.getInfo()
    #{'geodesic': False,
    # 'type': 'Polygon',
    # 'coordinates': [[[-137.147821, 58.823874], lower left
    #   [-137.147821, 58.845905], upper left
    #   [-137.090829, 58.845905], upper right
    #   [-137.090829, 58.823874], lower right
    #   [-137.147821, 58.823874]]]} ll again to close the polygon
    
    #Want to be able to easily paste back into LonMin,LonMax,LatMin,LatMax
    result = f"{rinfo['coordinates'][0][0][0]:.3f},{rinfo['coordinates'][0][2][0]:.3f},{rinfo['coordinates'][0][0][1]:.3f},{rinfo['coordinates'][0][1][1]:.3f}"
    print(result)
    #copy/paste result into glacierPropsLandsat.csv

In [ ]:
#export feature collection
geemap.ee_export_vector(fc, os.path.join(folder_land, "glacierPropsLandsat.shp"), verbose=True)

## reproject such that I get a rectangle in UTM8N

In [ ]:
def transform_regularize_bbox(wgs84_bbox):
    """
    Transform a WGS84 bounding box to UTM Zone 8N, regularize it to a rectangle,
    and transform it back to WGS84.
    
    Args:
        wgs84_bbox (list or tuple): [minx, miny, maxx, maxy] in WGS84 (lon, lat)
    
    Returns:
        tuple: Regularized bounding box [minx, miny, maxx, maxy] in WGS84
    """
    # Initialize Earth Engine (optional, only if visualization is needed)
    ee.Initialize()

    # Define CRS for WGS84 and UTM Zone 8N
    wgs84 = pyproj.CRS("EPSG:4326")
    utm7 = pyproj.CRS("EPSG:32607")
    utm8 = pyproj.CRS("EPSG:32608")
    
    # Create transformers
    project_to_utm = pyproj.Transformer.from_crs(wgs84, utm8, always_xy=True).transform
    project_to_wgs84 = pyproj.Transformer.from_crs(utm8, wgs84, always_xy=True).transform

    # Convert WGS84 bbox to Shapely geometry
    wgs84_box = box(wgs84_bbox[0], wgs84_bbox[1], wgs84_bbox[2], wgs84_bbox[3])

    # Transform to UTM Zone 8N
    utm8_box = transform(project_to_utm, wgs84_box)

    # Regularize to a rectangle in UTM
    utm8_rect = utm8_box.envelope

    #TODO: would be awesome to register corners to Landsat pixel corners, but that's too much for now.

    # Transform back to WGS84
    wgs84_rect = transform(project_to_wgs84, utm8_rect)
    wgs84_new_bounds = wgs84_rect.bounds  # (minx, miny, maxx, maxy)

#    return wgs84_new_bounds #NOTE: I actually want the wgs84_rect as the output, not wgs84_new_bounds
    return wgs84_rect

# Example usage
#wgs84_bbox = [-135.0, 57.0, -134.0, 58.0]
#wgs84_bbox_new = transform_regularize_bbox(wgs84_bbox)
#print("Original WGS84 BBox:", wgs84_bbox)
#print("Regularized WGS84 BBox:", wgs84_bbox_new)
#Result: Original WGS84 BBox: [-135.0, 57.0, -134.0, 58.0]
#Regularized WGS84 BBox: POLYGON ((-135 57.00000000000001, -134.0001071119298 56.996006464924804, -133.97228067948123 57.999779130695565, -135 58.003929218471185, -135 57.00000000000001))

In [ ]:
# Load the CSV file
df = pd.read_csv(file_path) #os.path.join(folder_base,'glacierPropsLandsat.csv'))
#df.iloc[0]

#Do the transformation (loop over glaciers)
transformed_regions = []
for index, row in df.iterrows():
    poly = transform_regularize_bbox([row['LonMin'], row['LatMin'], row['LonMax'], row['LatMax']])
    # Extract exterior coordinates
    coords=list(poly.exterior.coords)
    # Flatten coordinates into a single row with columns x1, y1, x2, y2, ...
    coord_dict = {}
    coord_dict['Name']=row['Name']
    for i, (x, y) in enumerate(coords, 1):
        coord_dict[f'x{i}'] = x
        coord_dict[f'y{i}'] = y
    transformed_regions.append(coord_dict)

print(transformed_regions[0:3])

In [ ]:
df.iloc[0,0:11]

In [ ]:
dft = pd.DataFrame(transformed_regions)
#pd.merge better than pd.concat([df, dft], axis=1) and df.join(dft). See test_proj.ipynb
#merge gets rid of any old transformed coordinates, only keeps WGS rectangle and new UTM-adjusted polygon
df_merge=pd.merge(df.iloc[:,0:10],dft,on='Name')
print(df_merge.iloc[0:3])

In [ ]:
#write to csv
df_merge.to_csv(file_path,index=False) #os.path.join(folder_base,'glacierPropsLandsat.csv')

In [ ]:
#add transformed boxes to map (add suffix _t)
# Initialize an empty list to store features
features_t = []

# Loop through each region in the CSV and add it to the map
for index, row in df_merge.iterrows():
    # Create a bounding box geometry
    region = ee.Geometry.Polygon([[row[f'x{i}'], row[f'y{i}']] for i in range(1, 6) ]) # x1,y1 to x5,y5
    # Create a feature with the region name
    feature_t = ee.Feature(region, {'name': row['Name']+" "+row['Region']})
    # Append to features list
    features_t.append(feature_t)
    # Add the region to the map with a unique color or style
    #Map.addLayer(feature_t, {'color': 'blue'}, row['Name']+" "+row['Region'])

# Convert the list to a FeatureCollection
fc_t = ee.FeatureCollection(features_t)
#NOTE: plotting works if we overwrite fc with fc.style() but saving no longer works

# Apply the styling to the FeatureCollection
fcs_t = fc_t.style(**{ #asterisk passes in as arguments instead of dictionary
    'color': '005500',      # green outline
    'width': 2,             # 2 pixels wide
    'lineType': 'dotted',
    'fillColor': '00000000' # 00000000 Fully transparent fill
    }) #(**style_params)

Map.addLayer(fcs_t, {}, 'All roi transformed')
Map